## DINOv2 LSTM ##

In [6]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from PIL import Image
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix


import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset, DataLoader


import os
import pickle
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image

## Device

In [7]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device = torch.device("cpu")
device

device(type='cuda')

## Load DinoV2

In [8]:
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
model.train()


Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0): NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
    (1): NestedT

In [9]:
# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
dataset = datasets.ImageFolder('/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256', transform) 
data_loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [11]:
from dinov2.loss import DINOLoss, iBOTPatchLoss, KoLeoLoss
dino_loss = DINOLoss(384)

In [ ]:
from torch.nn import functional as F

# Simple distillation loss example
def dino_loss(student_logits, teacher_logits, temperature=0.1):
    student_prob = F.softmax(student_logits / temperature, dim=-1)
    teacher_prob = F.softmax(teacher_logits / temperature, dim=-1)
    return F.kl_div(student_prob.log(), teacher_prob, reduction="batchmean")

In [ ]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for images, _ in data_loader:
    images = images.to(device)
    
    # Forward pass through the DINO model
    outputs = model(images)

    # Compute self-supervised loss here
    loss = dino_loss(outputs)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

NameError: name 'model' is not defined

In [ ]:
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# num_epochs = 3

# for epoch in range(num_epochs):
#     model.train()
#     for batch in train_dataloader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         optimizer.zero_grad()
        
#         outputs = model(**batch)
#         loss = outputs.loss
        
#         loss.backward()
#         optimizer.step()
    
#     print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {loss.item()}")
    
#     model.eval()
#     total_eval_loss = 0
#     for batch in eval_dataloader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         with torch.no_grad():
#             outputs = model(**batch)
#             total_eval_loss += outputs.loss.item()

In [ ]:
# ####### SECOND #######


frame_frequency = 2

# def create_label_dict(classes):
#     label_dict = {}
#     for i in range(0,len(classes)):
#         label_dict[classes[i]] = i
#     return label_dict

# class CustomImageDataset(Dataset):
#     def __init__(self, left_root_dir, right_root_dir):
        
#         left_pickle_file = open(left_root_dir, 'rb')
#         left_paths, left_features,left_labels = pickle.load(left_pickle_file)

#         right_pickle_file = open(right_root_dir, 'rb')
#         right_paths, right_features,right_labels = pickle.load(right_pickle_file)

#         self.left_features = left_features
#         self.right_features = right_features
#         self.paths = left_paths
#         self.classes = np.unique(left_labels)
#         label_dict = create_label_dict(self.classes)
#         self.labels = [label_dict[x] for x in left_labels]

#     def __len__(self):
#         return len(self.labels)
    
#     def __getitem__(self, idx):
#         splited_paths = self.paths[idx].split('/')

#         active_frame_indices = get_active_frames(splited_paths[-2],splited_paths[-1])
#         active_frame_indices = (
#             active_frame_indices
#             if active_frame_indices.size > 10
#             else np.arange(0, len(self.left_features[idx]))
#         )
#         left_embeddings = [self.left_features[idx][i] for i in active_frame_indices]
#         right_embeddings = [self.right_features[idx][i] for i in active_frame_indices]
#         left_embeddings = left_embeddings[0::frame_frequency]
#         right_embeddings = right_embeddings[0::frame_frequency]
#         embeddings = np.concatenate((left_embeddings, right_embeddings), axis=1)

#         np_stacked_array = np.stack(embeddings)
#         tensor = torch.from_numpy(np_stacked_array)
#         # trX = torch.stack(embeddings).float()
#         return tensor, self.labels[idx]
    

from torch.utils.data import Dataset, DataLoader

def create_label_dict(classes):
    label_dict = {}
    for i in range(0,len(classes)):
        label_dict[classes[i]] = i
    return label_dict

class CustomImageDataset(Dataset):
    def __init__(self, video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256"):

        full_sample_folder_list = []
        labels = []
        for label_folder in os.listdir(video_folder):
            full_label_folder = os.path.join(video_folder, label_folder)
            label = int(label_folder)
            # print("process_count: ", process_count, ' , label: ', label)
            for sample_folder in os.listdir(full_label_folder):
                full_sample_folder = os.path.join(full_label_folder, sample_folder)
                full_sample_folder_list.append(full_sample_folder)
                labels.append(label)

        self.classes = np.unique(labels)
        label_dict = create_label_dict(self.classes)
        self.labels = [label_dict[x] for x in labels]
        self.full_sample_folder_list = full_sample_folder_list

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        input_tensor_list = []
        full_sample_folder = self.full_sample_folder_list[idx]
        for image_file in os.listdir(full_sample_folder):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)

            input_tensor = transform(image)  # Add batch dimension
            input_tensor_list.append(input_tensor)
            # input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension
            # input_tensor_list.append(input_tensor)
        return input_tensor_list, self.labels[idx] 



In [ ]:
# image_dataset = CustomImageDataset()

# train_dataset, test_dataset = torch.utils.data.random_split(image_dataset, [0.85, 0.15])

train_dataset = CustomImageDataset()
test_dataset = CustomImageDataset()

cc = 5


In [ ]:
batch_size = 1
num_workers = 4

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class_names = train_dataset.classes
class_names

input_dim = train_dataset[0][0][0].size(0)  # Get input dimension from a single feature from a video
num_classes = len(set(train_dataset.classes))
print("input_dim: ", input_dim, " num_classes: ", num_classes)
print("train_dataset size: ", len(train_dataset))
print("test_dataset size: ", len(test_dataset))

input_dim:  3  num_classes:  744
train_dataset size:  22542
test_dataset size:  22542


## Model

In [ ]:
# class DinoVisionTransformerClassifier(nn.Module):
#     def __init__(self, input_dim, num_classes):
#         super(DinoVisionTransformerClassifier, self).__init__()
#         self.classifier = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.ReLU(),
#             nn.Linear(256, num_classes)
#         )
    
#     def forward(self, x):
#         x = self.classifier(x)
#         return x
    
# model = DinoVisionTransformerClassifier(input_dim=input_dim, num_classes=num_classes)
# model = model.to(device)


class VideoClassifierLSTM(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
        self.lstm = nn.LSTM(self.dino_model.embed_dim, hidden_dim, num_layers, batch_first=True, bidirectional=False, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # LSTM expects input shape: (batch, seq, features)
        features = []
        for input_tensor in x:
            feature = self.dino_model(input_tensor)
            features.append(feature)
        _, (hidden, _) = self.lstm(features)  # Use last hidden state
        output = self.dropout(hidden[-1])
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
hidden_dim = 216
num_layers = 2
model = VideoClassifierLSTM(hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes)
model = model.to(device)

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


In [ ]:
# class DinoVisionTransformerClassifier(nn.Module):
#     def __init__(self, input_dim, num_classes):
#         super(DinoVisionTransformerClassifier, self).__init__()
#         self.classifier = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.ReLU(),
#             nn.Linear(256, num_classes)
#         )
    
#     def forward(self, x):
#         x = self.classifier(x)
#         return x
    
# model = DinoVisionTransformerClassifier(input_dim=input_dim, num_classes=num_classes)
# model = model.to(device)


class VideoClassifierLSTM(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
        self.lstm = nn.LSTM(self.dino_model.embed_dim, hidden_dim, num_layers, batch_first=True, bidirectional=False, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # LSTM expects input shape: (batch, seq, features)
        features = []
        for input_tensor in x:
            feature = self.dino_model(input_tensor)
            features.append(feature)
        _, (hidden, _) = self.lstm(features)  # Use last hidden state
        output = self.dropout(hidden[-1])
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
hidden_dim = 216
num_layers = 2
model = VideoClassifierLSTM(hidden_dim=hidden_dim, num_layers=num_layers, num_classes=num_classes)
model = model.to(device)

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


## Functions

In [ ]:
def test_images():
    correct = 0
    top_5_correct = 0
    total = 0
    running_loss = 0.0
    # since we're not training, we don't need to calculate the gradients for our outputs
    test_predicted = []
    test_labels = []

    with torch.no_grad():
        for image_list, labels in test_loader:
            input_tensor_list = []
            for image in image_list:
                input_tensor = image.unsqueeze(0).to(device)  # Add batch dimension
                input_tensor_list.append(input_tensor)
        
            # features = features.to(device)
            labels = labels.to(device)

            # calculate outputs by running images through the network
            outputs = model(input_tensor_list)
            loss = criterion(outputs, labels)
            
            # the class with the highest energy is what we choose as prediction
            _, predicted = torch.topk(outputs.data, 1)
            _, predicted_top_5 = torch.topk(outputs.data, 5)
            total += labels.size(0)
            correct += (predicted.to(device) == labels).sum().item() 
            top_5_correct += (predicted_top_5.to(device) == labels).any().sum().item()
            running_loss += loss.item()

            test_labels += (labels.cpu().numpy().tolist())
            test_predicted += (predicted.cpu().numpy().tolist())

    avg_loss = running_loss / total
    accuracy = 100 * correct / total
    top_5_accuracy = 100 * top_5_correct / total
    print(f'Accuracy of the network on the {len(test_loader)*batch_size} test video: {accuracy:.4f} %, top5: {top_5_accuracy:.4f} %, avg_loss: {avg_loss}')
    return accuracy, top_5_accuracy, avg_loss

In [ ]:
import datetime
from time import gmtime, strftime
def get_current_time():
    return strftime("%Y-%m-%d_%H-%M-%S", gmtime())

def save_model_result(current_time):
    result_name = 'lstm_results/FINE_LSTM_RL_BI_' + current_time + '.pth'
    torch.save({'name': result_name,
                'model_state_dict': model.state_dict(),
                'lr': lr,
                'step_size': step_size,
                'gamma': gamma,
                'weight_decay': weight_decay,
                'hidden_dim': hidden_dim,
                'num_layers': num_layers,
                'batch_size': batch_size,
                'frame_frequency': frame_frequency,
                'input_dim': input_dim,
                'num_classes': num_classes,
                'train_dataset': len(train_dataset),
                'test_dataset': len(test_dataset),
                'avg_loss_list': avg_loss_list,
                'avg_accuracy_list': avg_accuracy_list,
                'avg_test_accuracy_list': avg_test_accuracy_list,
                'avg_top5_test_accuracy_list': avg_top5_test_accuracy_list,
                'avg_test_loss_list': avg_test_loss_list},
                result_name)



In [ ]:
import shutil
def copy_ipynb_file(current_time): 
    current_file = 'dinofine_tune_lstm_left.ipynb'
    copy_file = '/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/ipynbs/COPY_' + current_time + '_' + current_file
    shutil.copy(current_file, copy_file)

## Train

In [ ]:
lr = 0.0002
step_size = 10
gamma = 0.5
weight_decay = 0

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr)
scheduler = lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma) ## CosineAnnealingLR Dene
print(f"lr {lr}, step_size: {step_size}, gamma: {gamma}, weight_decay: {weight_decay}")
print(f"Model hidden_dim {hidden_dim}, num_layers: {num_layers}")
print(f"batch_size {batch_size}, frame_frequency: {frame_frequency}")

avg_loss_list = []
avg_accuracy_list = []
avg_test_accuracy_list = []
avg_top5_test_accuracy_list = []
avg_test_loss_list = []

num_epoch = 30
for epoch in range(num_epoch):
    train_acc = 0
    train_loss = 0
    loop = tqdm(train_loader)

    running_loss = 0.0
    running_accuracy= 0.0
    for idx, (image_list, labels) in enumerate(loop):
        input_tensor_list = []
        for image in image_list:
            input_tensor = image.to(device)  # Add batch dimension
            input_tensor_list.append(input_tensor)
        # features = features.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_tensor_list)
        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
        correct = (predictions == labels).sum().item()
        accuracy = correct / batch_size

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_accuracy += 100 * accuracy
        loop.set_description(f"Epoch [{epoch}/{num_epoch}]")
        loop.set_postfix(loss=loss.item(), acc=accuracy)
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    avg_accuracy = running_accuracy / len(train_loader)
    print(f"Time: {get_current_time()} Epoch [{epoch}], Avg loss: {avg_loss:.4f}, Avg accuracy: {avg_accuracy:.4f}")
    avg_test_accuracy, avg_top5_test_accuracy, avg_test_loss = test_images()

    avg_loss_list.append(avg_loss)
    avg_accuracy_list.append(avg_accuracy)
    avg_test_accuracy_list.append(avg_test_accuracy)
    avg_top5_test_accuracy_list.append(avg_top5_test_accuracy)
    avg_test_loss_list.append(avg_test_loss)
current_time = get_current_time()
save_model_result(current_time)
copy_ipynb_file(current_time)

lr 0.0002, step_size: 10, gamma: 0.5, weight_decay: 0
Model hidden_dim 216, num_layers: 2
batch_size 1, frame_frequency: 2


  0%|          | 0/22542 [00:00<?, ?it/s]

KeyboardInterrupt



In [ ]:
import matplotlib.pyplot as plt
import torch

# summarize history for accuracy
plt.plot(avg_accuracy_list) 
plt.plot(avg_test_accuracy_list)
plt.plot(avg_top5_test_accuracy_list)
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Test', 'Test Top-5'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(avg_loss_list)
plt.plot(avg_test_loss_list)
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

## Test

In [ ]:

# test_images()

## Report

In [ ]:
# print(classification_report(test_labels, test_predicted, target_names=class_names))


In [ ]:
# cm = confusion_matrix(test_labels, test_predicted)
# df_cm = pd.DataFrame(
#     cm, 
#     index = class_names,
#     columns = class_names
# )
# df_cm

In [ ]:
# def show_confusion_matrix(confusion_matrix):
#     hmap = sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="Blues")
#     plt.ylabel("Surface Ground Truth")
#     plt.xlabel("Predicted Surface")
#     plt.legend()
    
# show_confusion_matrix(df_cm)